In [ ]:
import ee
import pandas as pd
import numpy as np

ee.Authenticate()
try:
    ee.Initialize(project='ee-sakda-451407')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='ee-sakda-451407')

In [ ]:
# ---- config ----
# ช่วงเวลา: ส.ค. 2017 – ส.ค. 2025  (เท่า training period เดิมเป๊ะ)
# ข้อมูล NDVI + NDMI จาก MOD09A1 (500m, 8-day)
#   NDVI = (b02 - b01) / (b02 + b01)   [NIR=b02, Red=b01]
#   NDMI = (b02 - b06) / (b02 + b06)   [NIR=b02, SWIR1=b06]
#   scale factor: × 0.0001 ก่อนคำนวณ
# Hotspot จาก FIRMS T21 (MODIS) — ชุดเดียวกับ train
# Boundary: FAO GAUL 2015 — ชุดเดียวกับ train
# Output panel: Month | Province | hotspot | NDVI | NDMI

START = '2017-08-01'
END   = '2025-08-31'

GAUL = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))

PROVINCES = {
    'Chiang Mai':   GAUL.filter(ee.Filter.eq('ADM1_NAME', 'Chiang Mai')),
    'Mae Hong Son': GAUL.filter(ee.Filter.eq('ADM1_NAME', 'Mae Hong Son')),
    'Nan':          GAUL.filter(ee.Filter.eq('ADM1_NAME', 'Nan')),
    'Uttaradit':    GAUL.filter(ee.Filter.eq('ADM1_NAME', 'Uttaradit')),
}

# ตรวจสอบชื่อจังหวัด
for name, fc in PROVINCES.items():
    print(f'{name}: {fc.size().getInfo()} features')

In [ ]:
# ---- helper: build year-month list ----

def build_year_months(start, end):
    ym = []
    y, m = int(start[:4]), int(start[5:7])
    ey, em = int(end[:4]), int(end[5:7])
    while (y < ey) or (y == ey and m <= em):
        ym.append(y * 100 + m)
        m += 1
        if m > 12:
            m = 1
            y += 1
    return ym

In [ ]:
# ---- function: monthly panel (NDVI, NDMI, hotspot) for one province ----

def get_monthly_panel(start_date, end_date, study_area, province_name):
    print(f'  Loading MOD09A1 and FIRMS for {province_name}...')

    # MOD09A1: sur_refl_b01=Red, sur_refl_b02=NIR, sur_refl_b06=SWIR1
    mod09 = ee.ImageCollection('MODIS/061/MOD09A1') \
        .select(['sur_refl_b01', 'sur_refl_b02', 'sur_refl_b06']) \
        .filterDate(start_date, end_date) \
        .filterBounds(study_area)

    print(f'  MOD09A1 images: {mod09.size().getInfo()}')

    def add_indices(img):
        nir  = img.select('sur_refl_b02').multiply(0.0001)
        red  = img.select('sur_refl_b01').multiply(0.0001)
        swir = img.select('sur_refl_b06').multiply(0.0001)
        ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')
        ndmi = nir.subtract(swir).divide(nir.add(swir)).rename('NDMI')
        return img.addBands(ndvi).addBands(ndmi) \
            .copyProperties(img, ['system:time_start'])

    mod09_idx = mod09.map(add_indices)

    # FIRMS T21 (MODIS)
    firms = ee.ImageCollection('FIRMS') \
        .filterDate(start_date, end_date) \
        .filterBounds(study_area)

    print(f'  FIRMS images: {firms.size().getInfo()}')

    def to_presence(img):
        hp = img.select('T21').gt(0).rename('hotspot_presence')
        return img.addBands(hp).copyProperties(img, ['system:time_start'])

    firms_p = firms.map(to_presence)

    year_months = build_year_months(start_date, end_date)

    def make_monthly(ym):
        yr  = ee.Number(ym).divide(100).floor().int()
        mo  = ee.Number(ym).mod(100).int()
        st  = ee.Date.fromYMD(yr, mo, 1)
        en  = st.advance(1, 'month')

        # NDVI / NDMI monthly mean
        idx_col = mod09_idx.filterDate(st, en)
        idx_img = ee.Image(
            ee.Algorithms.If(
                idx_col.size().gt(0),
                idx_col.mean().select(['NDVI', 'NDMI']),
                ee.Image.constant([0, 0]).rename(['NDVI', 'NDMI'])
            )
        )

        # Hotspot monthly count
        hp_col = firms_p.filterDate(st, en)
        hp_val = ee.Number(
            ee.Algorithms.If(
                hp_col.size().gt(0),
                ee.Number(
                    hp_col.select('hotspot_presence').sum().reduceRegion(
                        reducer=ee.Reducer.sum(),
                        geometry=study_area,
                        scale=1000,
                        maxPixels=1e9
                    ).get('hotspot_presence', 0)
                ),
                0
            )
        )
        hp_img = ee.Image.constant(hp_val).rename('hotspot_count')

        combined = idx_img.addBands(hp_img)
        reduced  = combined.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=1000,
            maxPixels=1e9
        )
        return ee.Feature(None, reduced.combine({
            'date': st.format('YYYY-MM'),
            'year': yr,
            'month': mo
        }))

    fc   = ee.FeatureCollection(ee.List(year_months).map(make_monthly))
    rows = [f['properties'] for f in fc.getInfo()['features']]
    df   = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['year', 'month']).reset_index(drop=True)
        df['province'] = province_name
    return df

In [ ]:
# ---- ดึงข้อมูลทีละจังหวัด (อาจใช้เวลา 30-60 นาทีรวม) ----

all_dfs = []
for prov, sa in PROVINCES.items():
    print(f'\n=== {prov} ===')
    df_p = get_monthly_panel(START, END, sa, prov)
    if df_p.empty:
        print('  No data returned')
    else:
        print(df_p[['date', 'province', 'hotspot_count', 'NDVI', 'NDMI']].head(5).to_string(index=False))
        all_dfs.append(df_p)

print('\nAll provinces done.')

In [ ]:
# ---- สร้าง panel และ export ----

panel = pd.concat(all_dfs, ignore_index=True)
panel = panel[['date', 'province', 'hotspot_count', 'NDVI', 'NDMI']] \
    .rename(columns={'date': 'Month'})
panel = panel.sort_values(['Month', 'province']).reset_index(drop=True)

print(f'Panel shape: {panel.shape}')
print(panel.head(20).to_string(index=False))

# Summary stats
print('\n--- Correlation NDVI vs hotspot per province ---')
for prov in panel['province'].unique():
    sub = panel[panel['province'] == prov]
    r_ndvi = sub['NDVI'].corr(sub['hotspot_count'])
    r_ndmi = sub['NDMI'].corr(sub['hotspot_count'])
    print(f'{prov:15s}  NDVI-hotspot: {r_ndvi:+.3f}   NDMI-hotspot: {r_ndmi:+.3f}')

panel.to_csv('ndmi_panel_4provinces.csv', index=False)
panel.to_excel('ndmi_panel_4provinces.xlsx', index=False, sheet_name='NDMI_Panel')
print('\nSaved: ndmi_panel_4provinces.csv / .xlsx')